In [ ]:
# 既にインストールされていれば不要
!pip install quri-parts
!pip install "quri-parts[qulacs]"

In [2]:
from quri_parts.circuit import QuantumCircuit
from quri_parts.circuit import inverse_circuit
from quri_parts.circuit.utils.circuit_drawer import draw_circuit

n_controls = 5 # 制御量子ビットの数

def create_n_control_not_circuit(n_controls: int) -> QuantumCircuit:
    """
    トフォリゲートと n_controls -2 個の補助量子ビットを用いて多重制御NOTゲートの回路を作成する.
    インデックスは 0 ~ n_controls-1 が制御量子ビット, n_controls がターゲット量子ビット,
    n_controls+1 ~ n_qubits-1 が補助量子ビット.
    """
    assert n_controls >= 3, "n_controls must be at least 3"
    n_ancilla = n_controls - 2 # 補助量子ビットの数
    n_qubits = n_controls + n_ancilla + 1 # ターゲット量子ビットを含んだ全体の量子ビット数

    # 補助ビットを使ってAND を計算する量子回路
    computation_circuit = QuantumCircuit(n_qubits)
    computation_circuit.add_TOFFOLI_gate(0, 1, n_controls+1)
    for i in range(2, n_controls-1):
        computation_circuit.add_TOFFOLI_gate(i, n_controls+i-1, n_controls+i)

    # 多重制御NOT ゲートの量子回路の作成
    circuit = QuantumCircuit(n_qubits)
    circuit.extend(computation_circuit)
    # 補助ビットを使ってAND を計算した結果をターゲット量子ビットに反映
    circuit.add_TOFFOLI_gate(n_controls-1, n_qubits-1, n_controls)
    # uncomputation
    circuit.extend(inverse_circuit(computation_circuit))
    return circuit.freeze()

control_not_circuit = create_n_control_not_circuit(n_controls)
draw_circuit(control_not_circuit)

                                                        
                                                        
----●-----------------------------------------------●---
    |                                               |   
    |                                               |   
    |                                               |   
----●-----------------------------------------------●---
    |                                               |   
    |                                               |   
    |                                               |   
----|-------●-------------------------------●-------|---
    |       |                               |       |   
    |       |                               |       |   
    |       |                               |       |   
----|-------|-------●---------------●-------|-------|---
    |       |       |               |       |       |   
    |       |       |               |       |       |   
    |       |       |          

結果の見やすさのため、`input_bit`の`width`を書籍版に比べて1つ増やし、`input_bit`の左端が元々`0`であったことが分かるようにしました。

In [3]:
import numpy as np
from quri_parts.core.state import quantum_state
from quri_parts.qulacs.simulator import evaluate_state_to_vector

# すべての入力ビットを試す
for bit in range(2**n_controls):
    input_bit = np.binary_repr(bit, width=n_controls+1) # widthを変更
    state = quantum_state(2*n_controls-1, bits=bit, circuit=control_not_circuit)
    result_state_vector = evaluate_state_to_vector(state).vector
    # 波動関数に虚部がないか確認
    assert np.allclose(result_state_vector.imag, 0, atol=1e-8)
    # 波動関数の最大の値が1になっているか確認(=特定のビット状態の成分のみをもつ)
    max_output = np.argmax(np.abs(result_state_vector))
    assert np.isclose(result_state_vector[max_output], 1, atol=1e-8)
    # 出力のビット列を確認
    output_bit = np.binary_repr(max_output, width=n_controls+1)
    print(f"Input bit: {input_bit}, Output bit: {output_bit}")

Input bit: 000000, Output bit: 000000
Input bit: 000001, Output bit: 000001
Input bit: 000010, Output bit: 000010
Input bit: 000011, Output bit: 000011
Input bit: 000100, Output bit: 000100
Input bit: 000101, Output bit: 000101
Input bit: 000110, Output bit: 000110
Input bit: 000111, Output bit: 000111
Input bit: 001000, Output bit: 001000
Input bit: 001001, Output bit: 001001
Input bit: 001010, Output bit: 001010
Input bit: 001011, Output bit: 001011
Input bit: 001100, Output bit: 001100
Input bit: 001101, Output bit: 001101
Input bit: 001110, Output bit: 001110
Input bit: 001111, Output bit: 001111
Input bit: 010000, Output bit: 010000
Input bit: 010001, Output bit: 010001
Input bit: 010010, Output bit: 010010
Input bit: 010011, Output bit: 010011
Input bit: 010100, Output bit: 010100
Input bit: 010101, Output bit: 010101
Input bit: 010110, Output bit: 010110
Input bit: 010111, Output bit: 010111
Input bit: 011000, Output bit: 011000
Input bit: 011001, Output bit: 011001
Input bit: 0